In [26]:
import re
import copy
import json
import os
from collections import defaultdict

def remove_trailer(item):
    for c in 'abcdefg':
        if item.endswith(c):
            item = item[:-1]
    return item

transliterations = {}
signs = open('syllables.txt').readlines()
for s in signs:
    sign,value = s.split('\t')
    transliterations[sign] = value.strip()

numbers = {}
signs = open('numbers.txt').readlines()
for s in signs:
    sign,value = s.split('\t')
    numbers[sign] = value.strip()
    transliterations[sign] = value.strip()

lineara_signs = {v:k for k,v in transliterations.items()}
lineara_signs["*"] = "*"

json_file = open('inscriptions.json')
inscriptions = json.load(json_file)

json_file = open('supplement.json')
supplement = json.load(json_file)
catalog = {}
dimensions = {}
tabulations = {}
for s in supplement:
    catalog[s["name"]] = s["catalogue"]
    if s["dimensions"]:
        dimensions[s["name"]] = s["dimensions"]
    tabulations[s["name"]] = s["tabulation"]
        
commentaries = defaultdict(str)
for f in os.scandir("../commentary/"):
    if not f.is_file():
        continue
    content = open(f.path).read()
    nm = f.name.split('.')[0]
    commentaries[nm] = content
    for suff in 'abcdefg':
        commentaries[nm+suff] = content


In [43]:
contexts = {a:b for a,b in [
["EM", "Pre-palace period <br> 3500-1900 BCE"],
["EMI", "Early Minoan I <br> 3500-2900 BCE"],
["EMII", "Early Minoan IIA, IIB <br> 2900-2300 BCE"],
["EMIIIi", "Early Minoan III, Middle Minoan IA <br> 2300-1900 BCE"],
["MHIII", "Middle Helladic <br> 2000-1550 BCE"],
["MM", "Old Palace period <br>  1900-1650 BCE"],
["MMI", "Middle Minoan I  <br>  1900-1800 BCE"],
["MMIA", "Middle Minoan IA  <br>  1900-1850 BCE"],
["MMIB", "Middle Minoan IB  <br>  1900-1800 BCE"],
["MMII", "Middle Minoan II  <br>  1800-1650 BCE"],
["MMIIA", "Middle Minoan IIA  <br>  1800-1750 BCE"],
["MMIIB", "Middle Minoan IIB, IIIA <br>  1750-1650 BCE"],
["MMIII", "Middle Minoan III <br>  1750-1600 BCE"],
["MMIIIA", "Middle Minoan IIB, IIIA  <br> 1750-1650 BCE"],
["NP",  "New Palace period  <br> 1650-1450 BCE"],
["MMIIIB", "(first) Middle Minoan IIIB  <br>  1650-1600 BCE"],
["LMI", "Late Minoan I <br>  1600-1450 BCE"],
["LMIA", "Late Minoan IA <br>  1600-1500 BCE"],
["LMIB", "Late Minoan IB  <br>  1500-1450 BCE"],
["CM",  "Creto-Mycenaean period  <br>  1450-1100 BCE"],
["LHI",  "1550-1450 BCE"],
["LBI",  "1550-1400 BCE"],
["LMII", "Third Palace period, Late Minoan II, IIIA1  <br>  1450-1350 BCE"],
["LMIII",  "Post Palace period. Late Minoan IIIA2, IIIB, IIIC <br>  1400-1100 BCE"],
["LMIIIA",  "Post Palace period. Late Minoan IIIB  <br> 1350-1100 BCE"],
["SM",  "Sub-Minoan period  <br> 1100-1000 BCE"],
["Geometric",  "1000BCE"],
["","Not Available"]
]}
contexts


{'EM': 'Pre-palace period <br> 3500-1900 BCE',
 'EMI': 'Early Minoan I <br> 3500-2900 BCE',
 'EMII': 'Early Minoan IIA, IIB <br> 2900-2300 BCE',
 'EMIIIi': 'Early Minoan III, Middle Minoan IA <br> 2300-1900 BCE',
 'MHIII': 'Middle Helladic <br> 2000-1550 BCE',
 'MM': 'Old Palace period <br>  1900-1650 BCE',
 'MMI': 'Middle Minoan I  <br>  1900-1800 BCE',
 'MMIA': 'Middle Minoan IA  <br>  1900-1850 BCE',
 'MMIB': 'Middle Minoan IB  <br>  1900-1800 BCE',
 'MMII': 'Middle Minoan II  <br>  1800-1650 BCE',
 'MMIIA': 'Middle Minoan IIA  <br>  1800-1750 BCE',
 'MMIIB': 'Middle Minoan IIB, IIIA <br>  1750-1650 BCE',
 'MMIII': 'Middle Minoan III <br>  1750-1600 BCE',
 'MMIIIA': 'Middle Minoan IIB, IIIA  <br> 1750-1650 BCE',
 'NP': 'New Palace period  <br> 1650-1450 BCE',
 'MMIIIB': '(first) Middle Minoan IIIB  <br>  1650-1600 BCE',
 'LMI': 'Late Minoan I <br>  1600-1450 BCE',
 'LMIA': 'Late Minoan IA <br>  1600-1500 BCE',
 'LMIB': 'Late Minoan IB  <br>  1500-1450 BCE',
 'CM': 'Creto-Mycenaean p

In [27]:
def splitName(name):
    if len(name) < 4:
        return f"{name[:2]} {name[2:]}"
    if name[3] in "abcdfg":
        return f"{name[:2]} {name[2:4]} {name[4:]}"
    if name[3].isnumeric():
        return f"{name[:2]} {name[2:]}"
    if name.startswith("ARKH"):
        return f"{name[:4]} {name[4:]}"
    if len(name) < 5:
        return f"{name[:2]} {name[2:]}"
    if name[4] in "abcdfg":
        return f"{name[:3]} {name[3:5]} {name[5:]}"
    if name[:3] in ["VRY","THE","ARM","PYR"]:
        return f"{name[:3]} {name[3:]}"
    return name

split_names = {s[0]:splitName(s[0]) for s in inscriptions}

In [63]:

def get_image_elements(filenames,url):
    elements = ""
    for f in filenames:
        elements += f"<a href=\"../{url}\" target=\"_blank\"><img src=\"../{f}\" width=\"300\"></a>\n        "
    return elements

def createReadingSpec(name,tabulation, parsedInscription):
    run = ""
    line = 1
    spec = []
    for r,row in enumerate(tabulation):
        for sign, status, word_index in row:
            if sign not in transliterations:
                print(name,sign)
                s = "*"
            else:
                s = transliterations[sign]
            spec += [f"        {r+1} {line} {word_index} {s} {status if status else 'none'}"]
            run += sign
            if parsedInscription.startswith(run+'\n'):
                run += '\n'
                line+=1
    return spec

def createTranscribedAsciiFromSpec(spec):
    output = "       <line><word>"
    pr = None
    pw = None
    for s in spec:
        r,l,w,syl,status = s.split()
        if not pr:
            pr = r
            pw = w
        if w != pw:
            output += "</word>"
        if r != pr:
            pr = r
            output += "</line>\n       <line>"
        if w != pw:
            pw = w
            output += "<word>"
        if syl.isnumeric() and syl.isascii():
            output += f"<number>{syl}</number>"
        else:
            output += f"<ideogram>{syl}</ideogram>"
    output += "</word></line>"
    return output

def createParsedAsciiFromSpec(spec):
    output = "       <line><word>"
    pl = None
    pw = None
    for s in spec:
        r,l,w,syl,status = s.split()
        if not pl:
            pl = l
            pw = w
        if w != pw:
            output += "</word>"
        if l != pl:
            pl = l
            output += "</line>\n       <line>"
        if w != pw:
            pw = w
            output += "<word>"
        if syl.isnumeric() and syl.isascii():
            output += f"<number>{syl}</number>"
        else:
            output += f"<ideogram>{syl}</ideogram>"
    output += "</word></line>"
    return output

for inscription in inscriptions:
    name = inscription[0]
    detail = inscription[1]
    template = open("template.html").read()
    
    template = template.replace("%NAME%", name)
    template = template.replace("%NAMES%", ', '.join(detail["names"] + 
                                                    [split_names[name]]))

    template = template.replace("%CONTEXT%", contexts[detail["context"]])

    template = template.replace("%SITE%", detail["site"])
    template = template.replace("%SUPPORT%", detail["support"])

    findspot = detail["findspot"] if detail["findspot"] else "Unavailable"
    template = template.replace("%FINDSPOT%", findspot)

    scribe = detail["scribe"] if detail["scribe"] else "Unavailable"
    template = template.replace("%SCRIBE%", scribe)

    template = template.replace("%PHOTO_IMAGES%", 
                   get_image_elements(detail["images"], detail["imageRightsURL"]))
    template = template.replace("%PHOTO_SOURCE%", detail["imageRights"])
    
    template = template.replace("%DRAWING_IMAGES%", 
                   get_image_elements(detail["facsimileImages"],detail["imageRightsURL"]))
    template = template.replace("%DRAWING_SOURCE%", detail["imageRights"])
    
    ref_link = detail["imageRightsURL"]
    ref_url = f"<a href='{ref_link}'>{ref_link}</a>"
    template = template.replace("%REFERENCES%", ref_url)

    template = template.replace("%YOUNGER%", commentaries[name])
    template = template.replace("%CATALOG%", catalog[name] if name in catalog else "")
    
    if name in dimensions:
        template = template.replace("%HEIGHT%", dimensions[name]["height"])
        template = template.replace("%LENGTH%", dimensions[name]["length"])
        template = template.replace("%THICKNESS%", dimensions[name]["thickness"])
        template = template.replace("%SOURCE%", dimensions[name]["source"])
        template = template.replace("%UNIT%", dimensions[name]["unit"])
    else:
        template = template.replace("%HEIGHT%", "")
        template = template.replace("%LENGTH%", "")
        template = template.replace("%THICKNESS%", "")
        template = template.replace("%SOURCE%", "")
        template = template.replace("%UNIT%", "")
    
    if name in tabulations:
        tabulation = tabulations[name]
        parsedInscription = detail["parsedInscription"]
        parsedInscription = parsedInscription.strip("𐝫\n")
        parsedInscription = parsedInscription.replace("𐝫","")
        
        spec = createReadingSpec(name,tabulation,parsedInscription)
        template = template.replace("%SPEC%", '\n'.join(spec))
        
        transcribed_ascii = createTranscribedAsciiFromSpec(spec)
        template = template.replace("%TRANSCRIBED_ASCII%", transcribed_ascii)
        parsed_ascii = createParsedAsciiFromSpec(spec)
        template = template.replace("%PARSED_ASCII%", parsed_ascii)
        
        laspec = [' '.join(s.split()[:3] + [lineara_signs[s.split()[3]]] + [s.split()[4]])
                  for s in spec]
        transcribed_unicode = createTranscribedAsciiFromSpec(laspec)
        template = template.replace("%TRANSCRIBED_UNICODE%", transcribed_unicode)
        parsed_la = createParsedAsciiFromSpec(laspec)
        template = template.replace("%PARSED_UNICODE%", parsed_la)
        
    else:
        print(name)

    output_file = open(f"items/{name}.html",'w')
    output_file.write(template)
    output_file.flush()
    


HT17 󽇫
KHZc106
MYZf2 𐁁
PEZc4  
PEZc4 ≈
PEZc4  
PH11 |
PH11 |
PH11 |
PH11 |
PH11 |
PH11 |
PH11 |
PH11 |
PH11 |
PH11 |
PH11 |
PH11 |
PH11 |
PHWc46  
PHWc46 ≈
TELZb1  
TELZb1 ≈
TELZb1  
TRYZb1  
TRYZb1 ≈
TRYZb1  
ARGZg1  
ARGZg1 ≈
ARGZg1  
CRZg3  
CRZg3 ≈
CRZg3  
GOWc3 ,
PKZa28
KKHZb1
KH104
KH105
THEZg15
THEZg16
VRYZb2
VRYZb3
KNZg57a
KNZg57b
KNZg58
